![Health-Informatics: A Python Tutorial](../../Image/chapter-banner.png)


# Module 10: Clinical Natural Language Processing

**Health Informatics in Python** · Part III: Privacy, Text & Applied Informatics · Module 10 of 16

---

Roughly **80% of clinical information lives in free text** - notes, reports,
discharge summaries. Turning that text into structured, coded facts is the job of
**clinical NLP**. This module builds a rule-based clinical NLP pipeline: sentence
segmentation, concept extraction, **negation/context** detection, and mapping to
terminology codes.


## Learning objectives

By the end of this module you will be able to:

1. Explain why clinical text needs **specialized** NLP (abbreviations, negation, structure).
2. Tokenize and **sentence-segment** clinical notes with spaCy.
3. Extract **clinical concepts** with a dictionary/`PhraseMatcher` approach.
4. Detect **negation and context** (asserted vs negated vs historical/family) with a rule engine.
5. **Map** extracted mentions to SNOMED/RxNorm codes and detect **note sections**.


## Dataset

The two fabricated clinical notes from Module 9. No real patients; the point is the
*shape* of clinical language (abbreviations like "h/o", negations like "denies",
family-history mentions).


In [55]:
# Synthetic clinical notes containing FABRICATED identifiers (no real people)
notes = [
    {"note_id":"N001","patient_id":"P1000","text":
     "Ava Khan (MRN 4471982) seen on 03/12/2024. Contact 607-555-0148. "
     "58 yo F with h/o hypertension and type 2 diabetes presents for follow-up. "
     "BP 148/88. Denies chest pain. No shortness of breath. Continue lisinopril; "
     "recheck A1c in 3 months. Lives at 12 Elm St, Ithaca NY."},
    {"note_id":"N002","patient_id":"P1001","text":
     "Liam Ortiz, DOB 07/22/1969, MRN 5580321. Presents with acute bronchitis. "
     "Reports cough x5 days. Negative for fever. Family history of asthma. "
     "Started amoxicillin. Follow up if not improving. Email liam.o@example.com."},
]
for n in notes:
    print(n["note_id"], "-", n["text"][:70], "...")


N001 - Ava Khan (MRN 4471982) seen on 03/12/2024. Contact 607-555-0148. 58 yo ...
N002 - Liam Ortiz, DOB 07/22/1969, MRN 5580321. Presents with acute bronchiti ...


## 10.1 Why clinical NLP is hard

General-purpose NLP stumbles on clinical text because of:

- **Abbreviations & jargon** —-"h/o", "SOB", "BP", "x5 days".
- **Negation** - "denies chest pain" asserts the *absence* of a finding; naive keyword
  search would wrongly flag chest pain.
- **Context** - "family history of asthma" is about a *relative*, not the patient.
- **Structure** - notes have sections (HPI, PMH, Meds, Assessment) that change meaning.

A pipeline that ignores negation and context produces dangerously wrong structured data.


## 10.2 Tokenization and sentence segmentation

Tokenization and sentence segmentation are foundational pre-processing steps in NLP.
- **Tokenization** splits text into smaller units called tokens (typically words, numbers, punctuation).
  In clinical text, this must robustly handle abbreviations, medical terms, and identifiers (e.g., "h/o", "03/12/2024", "A1c").
- **Sentence segmentation** (sentence boundary detection) divides text into sentences.
  This can be tricky in clinical notes due to telegraphic style (missing periods), bulleted lists, or fragmentary findings.
spaCy can handle both steps and is easily extended for clinical nuances.

In [56]:
# This block demonstrates how to perform sentence segmentation (splitting text into sentences) with spaCy.
import spacy

# Try to load the small English spaCy model with NLP pipeline, which includes sentence segmenter.
try:
    nlp = spacy.load("en_core_web_sm")
except Exception:
    # If the model is unavailable (for example, not previously downloaded), 
    # create a blank English pipeline.
    nlp = spacy.blank("en")
    # If it lacks a sentence boundary detector ("sentencizer"), add it.
    if "sentencizer" not in nlp.pipe_names:
        nlp.add_pipe("sentencizer")

# Process the first clinical note to produce a spaCy Doc object.
doc = nlp(notes[0]["text"])

print("Sentences detected:")
# Iterate over the detected sentence spans and print each one.
for i, sent in enumerate(doc.sents, 1):
    print(f"  {i}. {sent.text.strip()}")

Sentences detected:
  1. Ava Khan (MRN 4471982) seen on 03/12/2024.
  2. Contact 607-555-0148.
  3. 58 yo F with h/o hypertension and type 2 diabetes presents for follow-up.
  4. BP 148/88.
  5. Denies chest pain.
  6. No shortness of breath.
  7. Continue lisinopril; recheck A1c in 3 months.
  8. Lives at 12 Elm St, Ithaca NY.


## 10.3 Concept extraction with a clinical gazetteer

A cornerstone technique in rule-based clinical NLP is the **dictionary lookup** approach. 
Here, a pre-defined, curated list of relevant clinical terms, known as a gazetteer, is matched directly to text using exact or near-exact string matches. 
This enables quick identification of medical problems, medications, symptoms, and other entities of interest.
spaCy's `PhraseMatcher` component is a highly efficient tool designed for this task:
- It allows for matching entire multi-word phrases (not just single tokens).
- Matching can be performed in a case-insensitive manner by specifying the appropriate attribute (like "LOWER").
- It is optimized for speed, which is important when scanning large sets of clinical notes.
With PhraseMatcher, the pipeline can rapidly flag the presence of terms from the clinical gazetteer, providing a foundation for further semantic analysis.


In [57]:
# This code demonstrates how to extract clinically relevant concepts—such as problems and medications—
# from clinical text using spaCy's PhraseMatcher with a simple gazetteer (dictionary lookup approach).

from spacy.matcher import PhraseMatcher

# Define a small clinical gazetteer (dictionary):
# It maps semantic type labels (like "problem" and "medication") to lists of term strings representing clinical entities.
GAZETTEER = {
    "problem": [
        "hypertension", "type 2 diabetes", "acute bronchitis", "asthma",
        "chest pain", "shortness of breath", "fever", "cough"
    ],
    "medication": [
        "lisinopril", "amoxicillin", "metformin", "atorvastatin"
    ],
}

# Initialize a PhraseMatcher to efficiently search for these multi-word phrases within a spaCy Doc,
# matching terms in a case-insensitive ("LOWER") way.
matcher = PhraseMatcher(nlp.vocab, attr="LOWER")

# For each semantic type and its associated terms, add the group of phrases to the matcher.
for semtype, terms in GAZETTEER.items():
    # nlp.make_doc(t) converts each phrase to a spaCy Doc object for accurate matching.
    matcher.add(semtype, [nlp.make_doc(t) for t in terms])

# Define a function to extract concepts from input text using the matcher:
def extract_concepts(text):
    # Process text into a spaCy Doc object.
    d = nlp(text)
    out = []
    # For every match found, extract info:
    for match_id, start, end in matcher(d):
        span = d[start:end]
        # Construct a dictionary for each concept match:
        out.append({
            "text": span.text,                            # matched phrase as-is in text
            "type": nlp.vocab.strings[match_id],          # "problem" or "medication"
            "start": span.start_char,                     # character offset where phrase starts
            "end": span.end_char,                         # character offset where phrase ends
            "sent": span.sent.text.strip()                # full sentence containing this match
        })
    # Return a list of concept matches (dictionaries) and the spaCy Doc object.
    return out, d

# Extract concepts from the first note in our clinical notes collection.
concepts, doc0 = extract_concepts(notes[0]["text"])

import pandas as pd
print("Concepts extracted from note N001:")
# Display the extracted concepts, showing only the phrase and its type.
print(pd.DataFrame(concepts)[["text", "type"]].to_string(index=False))

Concepts extracted from note N001:
               text       type
       hypertension    problem
    type 2 diabetes    problem
         chest pain    problem
shortness of breath    problem
         lisinopril medication


## 10.4 Negation and context detection

Extracting clinical concepts is only the first step: we also need to determine the context in which a concept appears.
For example, in sentences like "**Denies** chest pain" or "**no** shortness of breath", the clinical problem is actually *negated*—the patient does not have the problem.
To address this, we implement a simple **ConText-style** rule engine. This approach involves scanning the words before (and sometimes after) each concept in its sentence, searching for specific cue words or phrases—known as *triggers*—that change the assertion status of the concept.
Common triggers can indicate that a finding is negated (e.g., "no", "denies"), historical, or related to family history.
By detecting these triggers, we can assign each identified concept an assertion status, such as "present", "negated", "historical", or "family", thereby capturing more accurate clinical meaning from the text.


In [58]:
# Negation and context triggers for clinical concept assertion detection.
NEG_TRIGGERS    = ["denies", "no", "not", "negative for", "without", "rules out", "never"]       # Indicative of negation (the patient does NOT have the problem)
FAMILY_TRIGGERS = ["family history", "mother", "father", "sibling", "parent"]                   # Indicates the problem relates to family, not the patient
HIST_TRIGGERS   = ["h/o", "history of", "previous", "prior", "past"]                           # Indicates a historical problem, not currently present

def assert_status(concept, doc):
    """
    Determines the assertion status (e.g., present, negated, family, historical) of a clinical concept 
    within a document using simple keyword-based (ConText-style) rules.
    
    - Looks to the left of a concept mention in its sentence for cue words.
    - Returns a string indicating context status (family, negated, historical, or present).
    """
    sent = None
    # Find the sentence that contains the concept's mention based on character offsets.
    for s in doc.sents:
        if s.start_char <= concept["start"] < s.end_char:
            sent = s
            break
    # Examine the sentence fragment to the left of the mention, lowercased for easier matching.
    window = doc.text[sent.start_char:concept["start"]].lower() if sent else ""
    # Assign assertion status based on which trigger words are found in the window.
    if any(t in window for t in FAMILY_TRIGGERS):
        return "family"
    if any(t in window for t in NEG_TRIGGERS):
        return "negated"
    if any(t in window for t in HIST_TRIGGERS):
        return "historical"
    return "present"  # Default: the problem is actively present

# Apply assertion classification to each detected concept in the note.
for c in concepts:
    c["assertion"] = assert_status(c, doc0)

### Milestone 1 -  asserted vs negated problems

In clinical NLP, it's important to distinguish between problems that are actively present (asserted) and those that are specifically stated to be absent (negated) in the documentation.

*Asserted problems* are conditions explicitly documented as affecting the patient (e.g., "Patient has hypertension").

*Negated problems* are conditions the patient specifically denies or is documented as not having (e.g., "Denies chest pain"; "No shortness of breath").

This distinction is made possible by using contextual rules that look for negation cues (like "no", "denies") near mentions of clinical problems, ensuring more accurate extraction and interpretation of the patient's medical status.

In [59]:
# The following code block creates a DataFrame to organize all detected clinical concepts,
# displays each concept with its type and assertion status, and then summarizes which problems
# are currently active versus those explicitly marked as absent (negated).

# 1. Convert the list of concept dictionaries into a DataFrame for easier inspection/manipulation.
df = pd.DataFrame(concepts)

# 2. Print all concepts, showing their textual mention, their type (e.g., problem, medication), and their assertion status
#    (such as "present", "negated", etc.).
print("Concepts with assertion status (note N001):")
print(df[["text", "type", "assertion"]].to_string(index=False))

# 3. Extract and print the list of problems that are actively present (not negated, not family, not historical).
print("\nActive problems (present only):",
      df.query("type=='problem' and assertion=='present'")["text"].tolist())

# 4. Extract and print the list of concept mentions that have been classified as explicitly negated.
print("Explicitly negated       :",
      df.query("assertion=='negated'")["text"].tolist())
 

Concepts with assertion status (note N001):
               text       type  assertion
       hypertension    problem historical
    type 2 diabetes    problem historical
         chest pain    problem    negated
shortness of breath    problem    negated
         lisinopril medication    present

Active problems (present only): []
Explicitly negated       : ['chest pain', 'shortness of breath']


In [60]:
# This code demonstrates how the assertion classification identifies family-history and historical mentions in a clinical note (Note N002).
# 1. Extract clinical concepts and their surrounding spaCy document from the note text.
concepts2, doc2 = extract_concepts(notes[1]["text"])

# 2. For each detected concept, determine the assertion status (such as present, negated, family, or historical).
for c in concepts2:
    c["assertion"] = assert_status(c, doc2)

# 3. Build a DataFrame to organize and display all concepts along with their type and assertion results.
df2 = pd.DataFrame(concepts2)
print("Note N002 concepts:")
print(df2[["text","type","assertion"]].to_string(index=False))

# 4. Highlight the tagging of 'asthma' as a 'family' problem, indicating it belongs to a relative, not the patient themself.
print("\nNote 'asthma' is tagged 'family' — it belongs to a relative, not the patient.")

Note N002 concepts:
            text       type assertion
acute bronchitis    problem   present
           cough    problem   present
           fever    problem   negated
          asthma    problem    family
     amoxicillin medication   present

Note 'asthma' is tagged 'family' — it belongs to a relative, not the patient.


## 10.5 Mapping mentions to terminology codes

In the last step, we connect each clinical concept that is currently *present* in the note to a standardized terminology code
(using mapping dictionaries introduced earlier in Module 3). This transforms free-text extractions into structured, coded data
that can be consistently interpreted and analyzed across systems.


In [61]:
TERM_TO_CODE = {
    "hypertension":       ("SNOMED","38341003"),
    "type 2 diabetes":    ("SNOMED","44054006"),
    "acute bronchitis":   ("SNOMED","10509002"),
    "asthma":             ("SNOMED","195967001"),
    "lisinopril":         ("RxNorm","29046"),
    "amoxicillin":        ("RxNorm","723"),
    "metformin":          ("RxNorm","6809"),
}
def code_concept(text):
    return TERM_TO_CODE.get(text.lower(), (None,None))

present = df.query("assertion=='present'").copy()
present[["system","code"]] = present["text"].apply(
    lambda t: pd.Series(code_concept(t)))
print("Coded structured output from note N001 (present concepts):")
print(present[["text","type","assertion","system","code"]].to_string(index=False))

Coded structured output from note N001 (present concepts):
      text       type assertion system  code
lisinopril medication   present RxNorm 29046


## 10.6 Section detection

Clinical notes are divided into **sections**, each identified by a header such as "HPI", "Assessment", or "Medications".
The meaning of information found in a note often depends on which section it appears in—e.g., "Family History" indicates a condition applies to a relative, 
while "History of Present Illness" applies to the patient now.
To analyze notes accurately, it's important to detect and segment these sections.
A straightforward approach uses pattern matching to locate standard headers and split the note accordingly, 
so each segment can be interpreted with the proper clinical context.


In [62]:
SECTION_HEADERS = ["HPI","History of Present Illness","PMH","Past Medical History",
                   "Medications","Assessment","Plan","Family History"]
sample = ("HPI: 58 yo F with hypertension. "
          "Medications: lisinopril 10mg daily. "
          "Assessment: BP suboptimal, uptitrate.")
import re
parts = re.split(r"(" + "|".join(SECTION_HEADERS) + r")\s*:", sample)
sections = {parts[i].strip(): parts[i+1].strip() for i in range(1,len(parts)-1,2)}
for name, content in sections.items():
    print(f"[{name}] {content}")

[HPI] 58 yo F with hypertension.
[Medications] lisinopril 10mg daily.
[Assessment] BP suboptimal, uptitrate.


## Production tooling

This module intentionally uses simple rule-based Python, but real-world clinical NLP leverages more robust production-ready toolkits:

- **medspaCy**:  Adds clinical-specific extensions to spaCy, including a powerful and mature implementation of the ConText algorithm for asserting presence, negation, temporality, and experiencer within clinical text. It is designed for easy customization and integration into clinical workflows.
- **scispaCy**: — Provides spaCy pipelines trained on large biomedical corpora and includes functionality for fast named entity recognition (NER) and entity linking, especially mapping terms to standardized vocabularies like UMLS, SNOMED, or RxNorm.
- **Transformer models** (e.g., ClinicalBERT, GatorTron) — State-of-the-art neural network architectures pre-trained on large clinical corpora. These models achieve advanced context-sensitive understanding and are fine-tuned for tasks such as NER, assertion status detection, and contextual relation extraction. They enable extraction of subtle relationships and nuanced meanings in clinical documentation.

Regardless of toolkit, the underlying pipeline — *extract entities → determine assertion status → map to codes → segment into sections* — remains the same. Production NLP tools primarily differ in breadth of terminology handled, segmentation precision, negation/context accuracy, scalability, support for newer standards, and integration with medical ontologies.


In [63]:
# Example 1: medspaCy
# medspaCy extends spaCy with clinical sentence splitting and ConText (negation / historical / family).
# The default pipeline includes a target matcher, but you still have to tell it which concepts to find.
import medspacy
from medspacy.ner import TargetRule

nlp_medspacy = medspacy.load()
matcher = nlp_medspacy.get_pipe("medspacy_target_matcher")
matcher.add([
    TargetRule("MI", "PROBLEM"),
    TargetRule("hypertension", "PROBLEM"),
])

doc = nlp_medspacy("No history of MI. The patient has hypertension.")
for ent in doc.ents:
    print(f"Entity: {ent.text} | negated: {ent._.is_negated}")


2026-08-20 21:35:31.405 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=4] [doc 0] Token 0 'No' marked as sentence start (span begin)
2026-08-20 21:35:31.406 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=4] [doc 0] Token 5 'The' marked as sentence start (span end next token)
2026-08-20 21:35:31.407 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=4] [doc 0] Token 5 'The' marked as sentence start (span begin)
2026-08-20 21:35:31.407 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=4] Token/tag mapping: [(No, True), (history, False), (of, False), (MI, False), (., False), (The, True), (patient, False), (has, False), (hypertension, False), (., False)]


Entity: MI | negated: True
Entity: hypertension | negated: False


In [64]:
# Example 2: scispaCy
# Biomedical spaCy model trained on scientific/clinical text. Mentions are
# generic entity spans (drugs, diseases, procedures) rather than PERSON/GPE.
# Full UMLS linking is omitted here: the EntityLinker downloads a ~1 GB
# knowledge base on first use.
import spacy
import scispacy  # registers scispaCy tokenizers / components

nlp_scispacy = spacy.load("en_core_sci_sm")
doc2 = nlp_scispacy("Patient administered acetaminophen. History of diabetes mellitus.")

print("spaCy", spacy.__version__)
print("Biomedical mentions (en_core_sci_sm):")
for ent in doc2.ents:
    print(f"  {ent.text}")


spaCy 3.8.15
Biomedical mentions (en_core_sci_sm):
  Patient
  administered
  acetaminophen
  History
  diabetes mellitus


In [65]:

# Example 3: Transformer NER
# Raw ClinicalBERT (emilyalsentzer/Bio_ClinicalBERT) is a language model, not an
# NER tagger — loading it as TokenClassification randomly initializes the head.
# Use a model fine-tuned for biomedical token classification instead.
from transformers import pipeline

nlp_bert = pipeline(
    "token-classification",
    model="d4data/biomedical-ner-all",
    aggregation_strategy="simple",
)
text = "No chest pain. The patient is on metformin."
for res in nlp_bert(text):
    print(f"Entity: {res['word']} | type: {res['entity_group']} | score: {res['score']:.2f}")


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Entity: chest | type: Biological_structure | score: 1.00
Entity: met | type: Medication | score: 1.00


## Exercises

1. Add "SOB" and "h/o" expansions to the gazetteer/preprocessing so abbreviations
   are matched.
2. Improve the negation engine to stop at conjunctions ("but", "however") so scope
   doesn't leak across a sentence.
3. Extend `TERM_TO_CODE` and report the share of extracted concepts that were
   successfully coded (a coverage metric).



## Key takeaways

- Clinical NLP must handle **abbreviations, negation, context, and sections** — naive
  keyword search is unsafe.
- A practical pipeline is **segment → extract → assert → code → sectionize**.
- **Negation/context detection** is what separates "patient has X" from "patient
  denies X" — clinically decisive.



---
*Next: Module 11 — Clinical Decision Support Systems.*
